## Simple Example to usage of LLM response and parsing with Pydantic BaseModel

In [2]:
from pydantic import BaseModel, Field, ValidationError

class LLMResponseParser(BaseModel):
    """Schema for validating structured LLM outputs"""
    answer: str = Field(min_length=1, description="The LLM response")
    confidence_score: float = Field(ge=0.0,le=1.0)

# Dummy LLM response data in JSON format. Positive case
llm_reponse = {"answer":"RAG combines retrieval with generation.","confidence_score":0.90}
response = LLMResponseParser.model_validate(llm_reponse)
print(response.answer)
print(response.confidence_score)

llm_reponse = {"answer":"RAG combines retrieval with generation.","confidence_score":"NONE"}
try:
    response = LLMResponseParser.model_validate(llm_reponse)
    print(response.answer)
    print(response.confidence_score)
except ValidationError as ve:
    print(ve)



RAG combines retrieval with generation.
0.9
1 validation error for LLMResponseParser
confidence_score
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='NONE', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


## With Open AI API and Pydantic BaseModel

In [17]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError,field_validator, model_validator
from typing import Literal

class TicketAnalysis(BaseModel):
    # The LLM assigns category, priority, and confidence by reading the ticket and following your prompt. 
    # Pydantic does not decide their values—it only validates them afterward.
    category: Literal["billing","technical","account","other"]
    priority: Literal["low","medium","high"]
    confidence: float = Field(ge=0.0,le=1.0)
    # LLM response
    summary:str
    # LLM suggested response
    reply_draft:str

    @field_validator("priority",mode='before')
    @classmethod
    def validate_priority(cls, value):
        if value not in ["low","medium","high"]:
            raise ValueError("Priority should be in low,medium or high")
        return value

    # @model_validator validates the entire Pydantic model, not just one field.
    # Use it when correctness depends on the relationship between two or more fields.
    @model_validator(mode="after")
    def validate_confidence(self)->"TicketAnalysis":
        if self.priority == "high" and self.confidence < 0.70:
            raise ValueError("High-priority tickets require confidence of at least 0.70")
        return self

load_dotenv()

client = OpenAI()

input = [
    {
        "role":"user",
        "content":"I was charged twice for my subscription this month"
    }
]

# Parsing the response using TicketAnalysis Pydantic
response = client.responses.parse(
                model="gpt-5.4-mini",
                input=input,
                text_format=TicketAnalysis
            )
print(response.output_parsed)

category='billing' priority='high' confidence=0.99 summary='Customer reports being charged twice for their subscription this month, indicating a duplicate billing issue that needs review and likely refund processing.' reply_draft='Sorry about that — it looks like you may have been charged twice for your subscription this month. I can help look into it. Please send the email address on the account and, if possible, the dates and amounts of the charges. Once we verify the duplicate charge, we’ll help get it resolved as quickly as possible.'


## Usage Settings Management with BaseSettings

In [ ]:
import os
from pydantic_settings import BaseSettings
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

# Load the environment variables defined in .env file
load_dotenv()

# BaseSettings uses the environment variables
class OpenAIConfig(BaseSettings):
    openai_api_key:str 
    model_name: str = "gpt-5.4-mini"
    temperature:float = Field(default=0.0,ge=0.0,le=2.0)

config = OpenAIConfig()

# OpenAI client, get the api key from BaseSettings
client = OpenAI(api_key=config.openai_api_key)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":"What is OpenAI API?"}]
)
print(response)


ChatCompletion(id='chatcmpl-EEvdaJiTjdI7ut1cKS8xG7GJqHSfJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="The OpenAI API is a cloud-based service that allows developers to access and integrate OpenAI's advanced machine learning models, such as those used for natural language processing tasks. This API provides functionalities such as text generation, language translation, summarization, code generation, and more. Developers can utilize the API to enhance their applications with capabilities powered by models like GPT (Generative Pre-trained Transformer).\n\nKey features of the OpenAI API include:\n\n1. **Natural Language Understanding**: The API can interpret and respond to user inputs in a conversational manner.\n2. **Text Generation**: It can generate human-like text based on given prompts, making it useful for content creation, chatbots, and interactive experiences.\n3. **Customization**: Users can fine-tune the model on specifi